# 🔥 PyTorch Comprehensive Guide: From Tensors to Deep Learning

**A deeply detailed, from-scratch notebook covering the essentials of PyTorch: Tensors, Autograd, Neural Networks, Training, and Evaluation.**

---

## 📑 Table of Contents

| # | Topic | Key Concepts |
|---|-------|-------------|
| 1 | **Tensors: The Building Blocks** | Creation, datatypes, shapes, dimensions, CPU vs GPU |
| 2 | **Tensor Operations** | Reshaping, squeezing, slicing, matrix multiplication, broadcasting |
| 3 | **Autograd** | Automatic differentiation, computational graphs, gradients |
| 4 | **Neural Networks (`torch.nn`)** | `nn.Module`, layers, activation functions, sequential models |
| 5 | **Datasets & DataLoaders** | Handling data, batching, custom datasets |
| 6 | **Comprehensive Model Building** | Building a Multi-Layer Perceptron (MLP) and Convolutional Neural Network (CNN) |
| 7 | **Training Loop** | Optimizers, loss functions, forward pass, backward pass |
| 8 | **Evaluation & Saving Models** | Testing loops, metrics, saving and loading state dicts |

---

## 🔧 Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
# 1. 🧱 Tensors: The Building Blocks

A Tensor is a multi-dimensional matrix containing elements of a single data type. They are similar to NumPy arrays but with a superpower: **they can run on GPUs** to accelerate computing.

---

## 1.1 Creating Tensors
You can create tensors from Python lists, NumPy arrays, or use built-in PyTorch functions.

In [ ]:
# Creating tensors
# 1. From Python lists (0D, 1D, 2D, 3D)
tensor_0d = torch.tensor(42)                  # Scalar
tensor_1d = torch.tensor([1, 2, 3])           # Vector
tensor_2d = torch.tensor([[1, 2], [3, 4]])    # Matrix
tensor_3d = torch.tensor([[[1, 2], [3, 4]]])  # Tensor

print("--- Tensor Dimensions & Shapes ---")
print(f"0D Scalar: shape={tensor_0d.shape}, ndim={tensor_0d.ndim}")
print(f"1D Vector: shape={tensor_1d.shape}, ndim={tensor_1d.ndim}")
print(f"2D Matrix: shape={tensor_2d.shape}, ndim={tensor_2d.ndim}")
print(f"3D Tensor: shape={tensor_3d.shape}, ndim={tensor_3d.ndim}")

# 2. Using PyTorch built-in functions
zeros = torch.zeros(size=(2, 3))        # 2x3 matrix of zeros
ones = torch.ones(size=(3, 2))          # 3x2 matrix of ones
rand_t = torch.rand(size=(2, 2))        # Uniform distribution [0, 1)
randn_t = torch.randn(size=(2, 2))      # Normal distribution (mean=0, std=1)
arange_t = torch.arange(start=0, end=10, step=2)  # [0, 2, 4, 6, 8]
empty = torch.empty(size=(2, 2))        # Uninitialized data (whatever is in memory)

print("\n--- Built-in Functions ---")
print(f"Random Tensor:\n{rand_t}")
print(f"Arange Tensor:\n{arange_t}")

## 1.2 Datatypes and Devices
Tensors have specific data types (e.g., `torch.float32`, `torch.int64`) and reside on a specific device (`cpu` or `cuda`).

In [ ]:
# Datatypes
float_tensor = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
int_tensor = torch.tensor([1, 2, 3], dtype=torch.int64)
bool_tensor = torch.tensor([True, False, True], dtype=torch.bool)

print("--- Datatypes ---")
print(f"Float dtype: {float_tensor.dtype}")
print(f"Int dtype: {int_tensor.dtype}")

# Type casting
converted_tensor = int_tensor.type(torch.float32)
print(f"Converted dtype: {converted_tensor.dtype}")

# Devices
cpu_tensor = torch.tensor([1, 2, 3])
print("\n--- Devices ---")
print(f"Tensor is on device: {cpu_tensor.device}")

# Moving to GPU (if available)
if torch.cuda.is_available():
    gpu_tensor = cpu_tensor.to('cuda')
    print(f"Tensor moved to: {gpu_tensor.device}")
    # Note: Operations between CPU and GPU tensors will fail!
    # You must move them to the same device.

---
# 2. 🧮 Tensor Operations

Understanding how to manipulate tensor shapes and perform mathematical operations is crucial for building deep learning models.
---

## 2.1 Reshaping and Dimension Manipulation
- **`view()` / `reshape()`**: Change the shape of a tensor without changing its data.
- **`squeeze()`**: Removes all dimensions of size 1.
- **`unsqueeze()`**: Adds a dimension of size 1 at a specified position.
- **`permute()`**: Rearranges the dimensions of a tensor (useful for image channels).

In [ ]:
# Reshaping
x = torch.arange(1, 13)  # [1, 2, ..., 12], shape: (12,)
print(f"Original x: shape {x.shape}\n{x}")

x_reshaped = x.reshape(3, 4)  # Reshape to 3 rows, 4 columns
print(f"\nReshaped to (3, 4):\n{x_reshaped}")

x_view = x.view(2, 6) # View is similar to reshape, but guarantees sharing memory with original tensor if contiguous
print(f"\nViewed as (2, 6):\n{x_view}")

# Squeeze and Unsqueeze
y = torch.rand(size=(1, 3, 1, 4))
print(f"\nOriginal y shape: {y.shape}")

y_squeezed = y.squeeze()  # Removes all '1' dimensions
print(f"Squeezed y shape: {y_squeezed.shape}")

y_unsqueezed = y_squeezed.unsqueeze(dim=0)  # Add dimension at index 0
print(f"Unsqueezed y (dim=0) shape: {y_unsqueezed.shape}")

# Permute (Swapping dimensions)
# Common in CV: PyTorch uses (Batch, Channels, Height, Width)
# Matplotlib expects (Height, Width, Channels)
image_tensor = torch.rand(size=(3, 224, 224)) # C, H, W
image_permuted = image_tensor.permute(1, 2, 0) # Shift C to the end -> H, W, C
print(f"\nOriginal image shape (C, H, W): {image_tensor.shape}")
print(f"Permuted image shape (H, W, C): {image_permuted.shape}")

## 2.2 Slicing and Indexing
Accessing specific elements or sub-tensors works similarly to NumPy.

In [ ]:
x = torch.arange(1, 10).reshape(3, 3)
print(f"Matrix x:\n{x}")

# Get first row
print(f"\nFirst row: {x[0]}")

# Get first column
print(f"First column: {x[:, 0]}")

# Get a specific element (row 1, col 2)
print(f"Element at (1,2): {x[1, 2]}")

# Get a sub-matrix
print(f"\nSub-matrix (first two rows, last two columns):\n{x[:2, 1:]}")

## 2.3 Mathematical Operations & Broadcasting
- Element-wise operations (`+`, `-`, `*`, `/`)
- Matrix multiplication (`@` or `torch.matmul`)
- **Broadcasting**: PyTorch automatically expands smaller tensors to match larger ones for element-wise operations.

In [ ]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([10, 20, 30])

print("--- Element-wise Operations ---")
print(f"Addition: {a + b}")
print(f"Multiplication: {a * b}")

print("\n--- Broadcasting ---")
matrix = torch.zeros(3, 3)
vector = torch.tensor([1, 2, 3])
# Vector is broadcasted across all rows of the matrix
print(f"Matrix + Vector:\n{matrix + vector}")

print("\n--- Matrix Multiplication ---")
# Dot product (1D x 1D)
dot = torch.dot(a, b)
print(f"Dot product of {a} and {b} = {dot}")

# Matrix multiplication (2D x 2D)
mat1 = torch.tensor([[1, 2], [3, 4]])
mat2 = torch.tensor([[5, 6], [7, 8]])
matmul = torch.matmul(mat1, mat2)  # or mat1 @ mat2
print(f"\nMatmul of 2x2 matrices:\n{matmul}")

# Note: Inner dimensions must match! (m x n) @ (n x p) -> (m x p)

---
# 3. 📈 Autograd: Automatic Differentiation

Autograd is PyTorch's automatic differentiation engine. It records all operations performed on tensors that require gradients, creating a **computational graph**. When you call `.backward()`, it computes the gradients using the chain rule.
---


In [ ]:
# Enable gradient tracking on a tensor
x = torch.tensor(2.0, requires_grad=True)
print(f"x: {x}, requires_grad: {x.requires_grad}")

# Perform operations
y = x ** 2 + 5 * x
print(f"y (forward pass): {y}")

# Compute gradients (backward pass)
# dy/dx = 2x + 5. At x=2, dy/dx = 4 + 5 = 9
y.backward()

# The gradient is accumulated in x.grad
print(f"dy/dx evaluated at x=2: {x.grad}")

print("\n--- Stopping Gradient Tracking ---")
# Useful for evaluation/inference to save memory and compute
with torch.no_grad():
    z = x ** 2
    print(f"z requires_grad: {z.requires_grad}")

---
# 4. 🧠 Neural Networks (`torch.nn`)

`torch.nn` provides all the building blocks for creating neural networks: layers, activation functions, loss functions, etc. 

To build a custom model, you subclass `nn.Module` and define:
1. `__init__`: Define the layers.
2. `forward`: Define how data flows through the layers.
---


In [ ]:
# Example: A Simple Linear Regression Model using nn.Module
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear(in_features, out_features)
        # This creates weights (W) and bias (b) automatically
        self.linear_layer = nn.Linear(in_features=1, out_features=1)
        
    def forward(self, x):
        # Define the computation: y = Wx + b
        return self.linear_layer(x)

# Instantiate the model
model_0 = LinearRegressionModel()

# Inspect parameters
print("Model Parameters:")
for name, param in model_0.named_parameters():
    print(f"  {name}: {param.data}, requires_grad: {param.requires_grad}")

# Inference (Forward pass without training)
dummy_input = torch.tensor([[2.0]])
with torch.no_grad():
    output = model_0(dummy_input)
print(f"\nOutput for input 2.0: {output.item():.4f}")

---
# 5. 📦 Datasets and DataLoaders

- **`Dataset`**: Stores the samples and their corresponding labels.
- **`DataLoader`**: Wraps an iterable around the `Dataset` to enable easy access to the samples (handles batching, shuffling, multiprocessing).
---


In [ ]:
# Creating a custom Dataset
class ToyDataset(Dataset):
    def __init__(self, size=100):
        # Generate synthetic data: y = 3x + 2 + noise
        self.X = torch.rand(size, 1) * 10
        self.y = 3 * self.X + 2 + torch.randn(size, 1) * 0.5
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Instantiate dataset
dataset = ToyDataset(size=50)
print(f"Dataset size: {len(dataset)}")
print(f"First item: X={dataset[0][0].item():.4f}, y={dataset[0][1].item():.4f}")

# Create DataLoader
# batch_size: number of samples per batch
# shuffle: reshuffle data at every epoch (good for training)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Iterate through one batch
batch_X, batch_y = next(iter(dataloader))
print(f"\nBatch X shape: {batch_X.shape}")
print(f"Batch y shape: {batch_y.shape}")

---
# 6. 🏗️ Comprehensive Model Building

Let's build two common architectures: a Multi-Layer Perceptron (MLP) for tabular data/flat vectors, and a Convolutional Neural Network (CNN) for images.
---

## 6.1 Multi-Layer Perceptron (MLP)
Uses `nn.Sequential` for a cleaner syntax when layers just feed sequentially into each other.

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # nn.Sequential chains operations
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),           # Activation function introduces non-linearity
            nn.Dropout(p=0.2),   # Regularization: randomly zero out 20% of neurons during training
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        return self.network(x)

mlp_model = DeepMLP(input_dim=10, hidden_dim=32, output_dim=2)
print(mlp_model)

# Test forward pass
dummy_batch = torch.randn(5, 10)  # Batch of 5 samples, 10 features each
out = mlp_model(dummy_batch)
print(f"\nOutput shape: {out.shape} -> (batch_size, output_dim)")

## 6.2 Convolutional Neural Network (CNN)
Essential for spatial data like images. Uses `nn.Conv2d`, `nn.MaxPool2d`, and `nn.Flatten`.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Feature Extractor Block
        self.features = nn.Sequential(
            # Input: (Batch, Channels, Height, Width) -> e.g., (B, 1, 28, 28) for MNIST
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Halves spatial dimensions -> (B, 16, 14, 14)
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # Halves again -> (B, 32, 7, 7)
        )
        
        # Classifier Block
        self.classifier = nn.Sequential(
            nn.Flatten(), # Flattens (B, 32, 7, 7) into (B, 32 * 7 * 7)
            nn.Linear(in_features=32 * 7 * 7, out_features=128),
            nn.ReLU(),
            nn.Linear(in_features=128, out_features=num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = SimpleCNN()
print(cnn_model)

# Test forward pass
dummy_image_batch = torch.randn(8, 1, 28, 28) # Batch of 8 grayscale 28x28 images
out = cnn_model(dummy_image_batch)
print(f"\nOutput shape: {out.shape} -> (batch_size, num_classes)")

---
# 7. 🏃‍♂️ The Training Loop

The standard PyTorch training loop consists of 5 steps:
1. **Forward Pass**: Pass data through model `preds = model(x)`
2. **Calculate Loss**: Compare predictions to truth `loss = loss_fn(preds, y)`
3. **Zero Gradients**: Clear old gradients `optimizer.zero_grad()`
4. **Backward Pass**: Calculate gradients `loss.backward()`
5. **Optimizer Step**: Update weights `optimizer.step()`
---


In [ ]:
# Let's train the Linear Regression model on our ToyDataset

# 1. Setup Data
dataset = ToyDataset(size=200)
# Split into train/test (80/20)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 2. Setup Model, Loss, Optimizer
model = LinearRegressionModel().to(device)
loss_fn = nn.MSELoss() # Mean Squared Error for regression
optimizer = optim.SGD(model.parameters(), lr=0.01) # Stochastic Gradient Descent

# 3. Training Loop
epochs = 20
train_losses = []

for epoch in range(epochs):
    model.train() # Set model to training mode (enables gradients, dropout, etc.)
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader:
        # Move data to device
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # 1. Forward pass
        preds = model(batch_X)
        
        # 2. Calculate loss
        loss = loss_fn(preds, batch_y)
        
        # 3. Zero gradients
        optimizer.zero_grad()
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Optimizer step
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

print(f"\nLearned parameters:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.item():.4f}")
print("(Recall true relationship was y = 3x + 2)")

---
# 8. 📊 Evaluation & Saving Models

During evaluation, we turn off gradient tracking (`torch.no_grad()`) and set the model to eval mode (`model.eval()`).
---


In [ ]:
# Evaluation Loop
model.eval() # Set to eval mode (disables dropout, changes batch norm behavior)
test_loss = 0

with torch.no_grad(): # Turn off autograd engine for speed and memory
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)
        test_loss += loss.item()
        
avg_test_loss = test_loss / len(test_loader)
print(f"Test MSE Loss: {avg_test_loss:.4f}")

# Visualization
plt.figure(figsize=(10, 4))

# Plot 1: Loss Curve
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('MSE')

# Plot 2: Predictions vs Truth
plt.subplot(1, 2, 2)
X_plot = test_dataset.dataset.X[test_dataset.indices]
y_plot = test_dataset.dataset.y[test_dataset.indices]

model.to('cpu') # Move back to CPU for matplotlib
with torch.no_grad():
    preds_plot = model(X_plot)
    
plt.scatter(X_plot, y_plot, label='True Data', alpha=0.5, color='blue')
plt.plot(X_plot, preds_plot, label='Predictions', color='red')
plt.title('Model Predictions')
plt.legend()

plt.tight_layout()
plt.show()

## 8.2 Saving and Loading Models
The recommended way to save a model is to save its `state_dict`, which is a Python dictionary containing all learnable parameters.

In [ ]:
import os

# 1. Saving the model state_dict
save_path = 'my_linear_model.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

# 2. Loading the model
# Create a new instance of the model class
loaded_model = LinearRegressionModel()

# Load the state_dict into the new model
loaded_model.load_state_dict(torch.load(save_path))
loaded_model.eval() # Remember to set to eval mode!

print("\nLoaded Model Parameters:")
for name, param in loaded_model.named_parameters():
    print(f"  {name}: {param.item():.4f}")
    
# Clean up saved file
if os.path.exists(save_path):
    os.remove(save_path)

---
# 🎉 Summary

You now have a solid foundation in PyTorch!
1. **Tensors**: Multidimensional arrays on CPU/GPU.
2. **Autograd**: Automatic differentiation for calculating gradients.
3. **`nn.Module`**: Building blocks for architectures (MLPs, CNNs).
4. **Datasets/DataLoaders**: Efficient data handling.
5. **Training Loop**: Forward -> Loss -> Zero Grad -> Backward -> Step.
6. **Evaluation**: `torch.no_grad()` and `model.eval()`.
---
